# AI Risk Management

**Use case:** Score, classify, treat, and document AI risks.


**Total steps:** 10

## Installation

```bash
pip install pandas numpy  matplotlib langchain-openai python-dotenv
```

## Step 1 - Load the AI risk register

In [ ]:
import pandas as pd
from pathlib import Path
df = pd.read_csv(Path('data/ai_risk_register.csv'))
df


## Step 2 - Calculate inherent risk score

In [ ]:
df['inherent_score'] = df['impact']*df['likelihood']
print(df[['risk_id','risk_name','inherent_score']])


## Step 3 - Convert scores to bands

In [ ]:
def band(score):
    if score>=16:return 'Critical'
    if score>=10:return 'High'
    if score>=5:return 'Medium'
    return 'Low'
df['calculated_band'] = df['inherent_score'].apply(band)
print(df[['risk_name','calculated_band']])


## Step 4 - Add control effectiveness

In [ ]:
control_effectiveness = {'Strong':0.60,'Moderate':0.40,'Weak':0.20}
df['control_effectiveness'] = ['Strong','Strong','Moderate','Moderate','Strong','Strong','Moderate','Moderate','Strong','Moderate']


## Step 5 - Calculate residual risk

In [ ]:
df['residual_score'] = (df['inherent_score']*(1-df['control_effectiveness'].map(control_effectiveness))).round(1)
print(df[['risk_name','inherent_score','residual_score']])


## Step 6 - Identify high residual risks

In [ ]:
high_residual = df[df['residual_score']>=8]
print(high_residual[['risk_name','residual_score']])


## Step 7 - Assign risk owners

In [ ]:
owner_map = {'Privacy':'Privacy Lead','Security':'Security Lead','Reliability':'AI Engineering','Fairness':'Responsible AI Lead','Safety':'Safety Lead','Transparency':'Model Owner','Operations':'MLOps','Governance':'GRC','Supply Chain':'Vendor Risk'}
df['owner'] = df['domain'].map(owner_map).fillna('AI Governance')


## Step 8 - Define treatment decision

In [ ]:
df['treatment'] = np.where(df['residual_score']>=8,'MITIGATE','MONITOR')
print(df[['risk_name','owner','treatment']])


## Step 9 - Create a risk summary

In [ ]:
summary = df.groupby('treatment').size().to_dict()
print(summary)


## Step 10 - Save governance evidence

In [ ]:
df.to_csv('completed_ai_risk_assessment.csv',index=False)
print('Saved completed_ai_risk_assessment.csv')
